# Benchmark Workbench

We want to build a benchmark that measures something **new** about LLMs.

The loop:
1. Write a `@task` — a function that prompts an LLM and scores its response
2. Run it on several models to get scores  
3. Ask BenchPress: "can you predict these scores from existing benchmarks?"
4. If yes → our benchmark is redundant. If no → it measures something new.

## Setup

In [8]:
import os
import hashlib
import numpy as np

from dotenv import load_dotenv
from cachy import enable_cachy

from evaluating_agi.benchpress import check_novelty, MODEL_IDX

enable_cachy()
load_dotenv()


True

The **kaggle-benchmarks SDK** gives us `@task` (decorator to define a benchmark)
and `llm.prompt()` (to call a model). It talks to any OpenAI-compatible API.

One gotcha: the SDK reads env vars at import time to create a default model handle.
So we set them first, then import.

In [9]:
# SDK reads these at import time
os.environ["MODEL_PROXY_URL"]     = "https://api.openai.com/v1"
os.environ["MODEL_PROXY_API_KEY"] = os.environ["OPENAI_API_KEY"]
os.environ["LLM_DEFAULT"]         = "gpt-4.1-mini"

from kaggle_benchmarks import llm, task
from kaggle_benchmarks.kaggle.model_proxy import ModelProxy
print(f"Default model: {llm.name}")

Default model: gpt-4.1-mini


To run a task on **different** models, we create LLM handles via `ModelProxy`.

One gotcha: the SDK sends `seed=0` for reproducibility. OpenAI accepts it,
but Gemini and xAI reject it with a 400 error. So we patch it out for those.

In [10]:
PROVIDERS = {
    "openai": ("https://api.openai.com/v1",                              "OPENAI_API_KEY"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "GEMINI_API_KEY"),
    "xai":    ("https://api.x.ai/v1",                                    "XAI_API_KEY"),
}

def get_llm(provider, model):
    base_url, key_env = PROVIDERS[provider]
    m = ModelProxy(model=model, api_key=os.environ[key_env], base_url=base_url)
    if provider in ("gemini", "xai"):  # patch out seed
        _orig = m.invoke
        def patched(messages, system=None, **kw): kw.pop("seed", None); return _orig(messages, system=system, **kw)
        m.invoke = patched
    return m

## Models

For BenchPress to work, we need models that are **rows in its 83×49 matrix**.
Each entry maps: provider → API model name → BenchPress ID.

In [11]:
EVAL_MODELS = [
    ("openai",  "gpt-4.1-mini",    "gpt-4.1-mini"),      # cheap
    ("gemini",  "gemini-2.5-flash", "gemini-2.5-flash"),  # cheap, different provider
    ("xai",     "grok-3-beta",      "grok-3-beta"),       # mid
    ("openai",  "gpt-4.1",          "gpt-4.1"),           # strong
    ("openai",  "o4-mini",          "o4-mini-high"),       # reasoning model
]

# Sanity check: all BenchPress IDs exist in the matrix
for _, _, bp_id in EVAL_MODELS: assert bp_id in MODEL_IDX, f"{bp_id} not in matrix!"
print(f"{len(EVAL_MODELS)} models, all in BenchPress matrix ✓")

5 models, all in BenchPress matrix ✓


## The pipeline

`run_on_models` runs a task on each model and returns scores in BenchPress format (0-100).
`evaluate_and_check` = run + novelty check in one call.

In [12]:
def run_on_models(task_fn, models=None, n_trials=1):
    """Run a @task on each model → {benchpress_id: score_0_to_100}."""
    results = {}
    for provider, api_name, bp_id in (models or EVAL_MODELS):
        trial_scores = []
        for _ in range(n_trials):
            try:
                run = task_fn.run(get_llm(provider, api_name))
                s = float(run.result) if isinstance(run.result, (int, float)) else (1.0 if run.passed else 0.0)
            except Exception as e:
                print(f"  ✗ {api_name}: {e}"); s = 0.0
            trial_scores.append(s)
        avg = np.mean(trial_scores) * 100
        results[bp_id] = avg
        print(f"  {api_name:<25s} → {avg:5.1f}%  (trials: [f{s*100:.0f} for s in trial_scores])")
    return results

def evaluate_and_check(task_fn, name, models=None, n_trials=1):
    """Run on models → check novelty with BenchPress."""
    print(f"\n🔬 {name}\n" + "─"*50)
    scores = run_on_models(task_fn, models, n_trials)
    return check_novelty(scores, name)

## Smoke test: a deliberately meaningless benchmark

Before building real benchmarks, lets validate the pipeline with a **random** one.

The task below asks 10 trivial questions (so we actually call the LLM), but the
score is a coin flip based on `md5(model_name + question_index)`. Each model gets
a different deterministic-but-meaningless score.

**What we expect**: BenchPress should *fail* to predict these scores (they have no
relationship to model capability), giving a high median error (~18+ pts).

In [13]:
@task("random_benchmark", description="Scores by coin flip — should look like noise to BenchPress")
def random_benchmark(llm) -> float:
    questions = [
        "What is 2+2?", "Name a primary color.", "What planet is closest to the Sun?",
        "Is water wet?", "What comes after Monday?", "Spell the word cat.",
        "What is the boiling point of water in Celsius?", "Name a mammal.",
        "How many sides does a triangle have?", "What language is spoken in France?",
    ]
    correct = 0
    for i, q in enumerate(questions):
        llm.prompt(q)  # actually call the LLM (we ignore the answer)
        h = int(hashlib.md5(f"{llm.name}_{i}".encode()).hexdigest(), 16)
        if h % 2 == 0: correct += 1  # coin flip, seeded by model+question
    return correct / len(questions)

In [14]:
# check_novelty needs ≥3 models. Use the 3 cheapest.
CHEAP = EVAL_MODELS[:3]
evaluate_and_check(random_benchmark, "Random benchmark (expect: ~18+ pts)", CHEAP)


🔬 Random benchmark (expect: ~18+ pts)
──────────────────────────────────────────────────
  gpt-4.1-mini              →  60.0%  (trials: [f60 for s in trial_scores])
  gemini-2.5-flash          →  70.0%  (trials: [f70 for s in trial_scores])
  grok-3-beta               →  40.0%  (trials: [f40 for s in trial_scores])

═══════════════════════════════════════════════════════
  Random benchmark (expect: ~18+ pts)
═══════════════════════════════════════════════════════
  Model                           Actual    Pred   Error
  -----------------------------------------------------
  Grok 3 Beta                       40.0    68.1    28.1  ← big miss
  Gemini 2.5 Flash                  70.0    46.8    23.2  ← big miss
  GPT-4.1 mini                      60.0    58.4     1.6

  Median LOO error: 23.2 pts
  → ✓ Looks NOVEL (hard to predict from existing evals)
═══════════════════════════════════════════════════════



{'name': 'Random benchmark (expect: ~18+ pts)',
 'median_error': np.float64(23.191372605648162),
 'results': [('gpt-4.1-mini',
   np.float64(60.0),
   np.float64(58.39302851281219),
   np.float64(1.6069714871878134)),
  ('gemini-2.5-flash',
   np.float64(70.0),
   np.float64(46.80862739435184),
   np.float64(23.191372605648162)),
  ('grok-3-beta',
   np.float64(40.0),
   np.float64(68.08749078868239),
   np.float64(28.087490788682388))]}

## Reading the results

The report shows, for each model:
- **Actual**: the score our benchmark gave it
- **Pred**: what BenchPress predicted from the models other 49 benchmark scores
- **Error**: |Actual - Pred|

**Median LOO error** is the key number:
- **< 5** → redundant (BenchPress already knows this from existing benchmarks)
- **5–10** → partially novel
- **12+** → novel (measures something BenchPress cant predict)
- **~18+** → noise territory (no signal at all)

Our random benchmark should land in noise territory. If it does, the pipeline works
and we can trust it to tell us when a *real* benchmark is novel.